# 🚀 Qwen Model Optimization Pipeline

**Download → Convert → Compress → Distill → Compress → Optimize → Deploy**

This notebook implements a complete pipeline for running Qwen 2.5 (and Qwen 3.6) models
on Google Colab with minimal VRAM and near-instantaneous inference.

Grounded in formally verified compression theory (Lean 4 proofs).

| Stage | Technique | Theory Source |
|-------|-----------|---------------|
| Quantization | AWQ/GPTQ 4-bit | `QuantizationBounds.lean` |
| Pruning | Wanda 50% | `PruningBounds.lean` |
| Distillation | KD with T=4 | `DistillationTheory.lean` |
| Inference | Speculative decoding | `SpeculativeDecodingTheory.lean` |
| Composition | Error/ratio bounds | `CompressionPipeline.lean` |

## 0. Setup & Mount Google Drive

In [ ]:
# Mount Google Drive for model caching / checkpointing
from google.colab import drive
drive.mount('/content/drive')

# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q torch transformers>=4.45.0 accelerate safetensors
!pip install -q autoawq auto-gptq optimum bitsandbytes
!pip install -q vllm datasets sentencepiece protobuf
!pip install -q psutil gputil tqdm huggingface_hub

In [ ]:
# Upload and import the pipeline
# Option A: Clone the repo
# !git clone <your-repo> /content/pipeline

# Option B: Copy from Drive
# !cp /content/drive/MyDrive/pipeline/*.py /content/

# Option C: Use inline (this notebook is self-contained)
import sys
sys.path.insert(0, '/content')

## 1. Configure the Pipeline

In [ ]:
# ====================================================================
# CONFIGURATION — Edit this cell to change models and settings
# ====================================================================

# Choose your model:
#   'qwen2.5-0.5b'     — Tiny, runs on free Colab T4
#   'qwen2.5-1.5b'     — Small, runs on free Colab T4
#   'qwen2.5-7b'       — Medium, needs Colab Pro (T4/A100)
#   'qwen2.5-14b'      — Large, needs A100
#   'qwen2.5-72b'      — Very large, needs A100 80GB
#   'qwen3.6-35b-a3b'  — MoE, ~3B active params, runs on T4!

MODEL_CHOICE = 'qwen2.5-7b'  # Start here, then try qwen3.6-35b-a3b

# Quantization
QUANT_METHOD = 'awq'   # awq | gptq | gguf
QUANT_BITS = 4         # 4 | 8

# Pruning
PRUNING_SPARSITY = 0.50  # 0.0 to 0.70

# Distillation (set True for 7B+ models)
DISTILL_ENABLED = False

# Framework
FRAMEWORK = 'vllm'     # vllm | transformers | llamacpp

print(f'Model: {MODEL_CHOICE}')
print(f'Quantization: {QUANT_METHOD} {QUANT_BITS}-bit')
print(f'Pruning: {PRUNING_SPARSITY*100:.0f}%')
print(f'Framework: {FRAMEWORK}')

## 2. Download & Cache Model

In [ ]:
import os, time, json, shutil
from huggingface_hub import snapshot_download

MODEL_IDS = {
    'qwen2.5-0.5b': 'Qwen/Qwen2.5-0.5B-Instruct',
    'qwen2.5-1.5b': 'Qwen/Qwen2.5-1.5B-Instruct',
    'qwen2.5-7b': 'Qwen/Qwen2.5-7B-Instruct',
    'qwen2.5-14b': 'Qwen/Qwen2.5-14B-Instruct',
    'qwen2.5-72b': 'Qwen/Qwen2.5-72B-Instruct',
    'qwen3.6-35b-a3b': 'Qwen/Qwen3.6-35B-A3B',
}

MODEL_ID = MODEL_IDS[MODEL_CHOICE]
LOCAL_PATH = f'/content/models/{MODEL_CHOICE}'
GDRIVE_PATH = f'/content/drive/MyDrive/qwen_cache/{MODEL_CHOICE}'

# Check Google Drive cache first
if os.path.exists(GDRIVE_PATH) and os.listdir(GDRIVE_PATH):
    print(f'✅ Found cached model in Google Drive: {GDRIVE_PATH}')
    if not os.path.exists(LOCAL_PATH):
        print('Copying to local storage...')
        shutil.copytree(GDRIVE_PATH, LOCAL_PATH)
    print('Cache restored!')
else:
    print(f'⬇️ Downloading {MODEL_ID} from HuggingFace...')
    os.makedirs(LOCAL_PATH, exist_ok=True)
    start = time.time()
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=LOCAL_PATH,
        local_dir_use_symlinks=False,
        resume_download=True,
    )
    elapsed = time.time() - start
    print(f'Downloaded in {elapsed:.1f}s')

    # Cache to Google Drive
    print('💾 Caching to Google Drive for future sessions...')
    os.makedirs(GDRIVE_PATH, exist_ok=True)
    shutil.copytree(LOCAL_PATH, GDRIVE_PATH, dirs_exist_ok=True)
    print('Cached!')

# Report size
def dir_size_gb(path):
    total = sum(os.path.getsize(os.path.join(dp, f))
               for dp, _, fns in os.walk(path) for f in fns)
    return total / (1024**3)

print(f'\n📦 Model size: {dir_size_gb(LOCAL_PATH):.2f} GB')

## 3. Quantize (Compress #1)

Apply AWQ/GPTQ 4-bit quantization.

**Theory** (from `QuantizationBounds.lean`):
- Per-weight error: `|x − Q(x)| ≤ δ/2` where `δ = range/2^bits`
- Frobenius bound: `‖W−Q(W)‖_F ≤ (δ/2)√(nm)`

In [ ]:
import torch
import math

QUANT_OUTPUT = f'/content/models/{MODEL_CHOICE}-{QUANT_METHOD}-{QUANT_BITS}bit'

# Log theoretical bounds
weight_range = 2.0
delta = weight_range / (2 ** QUANT_BITS)
print(f'📐 Theoretical bounds ({QUANT_BITS}-bit):')
print(f'   Step size δ = {delta:.6f}')
print(f'   Per-weight error ≤ {delta/2:.6f}')
print(f'   Perplexity factor ≤ e^(δ/2) = {math.exp(delta/2):.6f}')
print()

if QUANT_METHOD == 'awq':
    from awq import AutoAWQForCausalLM
    from transformers import AutoTokenizer

    print(f'🔧 AWQ {QUANT_BITS}-bit quantization...')
    tokenizer = AutoTokenizer.from_pretrained(LOCAL_PATH, trust_remote_code=True)
    model = AutoAWQForCausalLM.from_pretrained(LOCAL_PATH, trust_remote_code=True)

    quant_config = {
        'zero_point': True,
        'q_group_size': 128,
        'w_bit': QUANT_BITS,
        'version': 'GEMM',
    }
    model.quantize(tokenizer, quant_config=quant_config)

    os.makedirs(QUANT_OUTPUT, exist_ok=True)
    model.save_quantized(QUANT_OUTPUT)
    tokenizer.save_pretrained(QUANT_OUTPUT)

elif QUANT_METHOD == 'gptq':
    from transformers import AutoModelForCausalLM, AutoTokenizer, GPTQConfig

    print(f'🔧 GPTQ {QUANT_BITS}-bit quantization...')
    tokenizer = AutoTokenizer.from_pretrained(LOCAL_PATH, trust_remote_code=True)
    gptq_config = GPTQConfig(
        bits=QUANT_BITS, group_size=128, desc_act=True,
        dataset='wikitext2', tokenizer=tokenizer,
    )
    model = AutoModelForCausalLM.from_pretrained(
        LOCAL_PATH, quantization_config=gptq_config,
        device_map='auto', trust_remote_code=True,
    )
    os.makedirs(QUANT_OUTPUT, exist_ok=True)
    model.save_pretrained(QUANT_OUTPUT)
    tokenizer.save_pretrained(QUANT_OUTPUT)

else:  # BitsAndBytes fallback
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    print(f'🔧 BitsAndBytes {QUANT_BITS}-bit quantization...')
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=(QUANT_BITS == 4),
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
    )
    tokenizer = AutoTokenizer.from_pretrained(LOCAL_PATH, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        LOCAL_PATH, quantization_config=bnb_config,
        device_map='auto', trust_remote_code=True,
    )
    QUANT_OUTPUT = LOCAL_PATH  # BnB loads in-place

orig_gb = dir_size_gb(LOCAL_PATH)
quant_gb = dir_size_gb(QUANT_OUTPUT)
print(f'\n✅ Quantization complete!')
print(f'   Original: {orig_gb:.2f} GB → Quantized: {quant_gb:.2f} GB')
print(f'   Compression ratio: {orig_gb/max(quant_gb, 0.01):.1f}×')

## 4. Prune (Compress #2)

Apply Wanda-style magnitude pruning for additional compression.

**Theory** (from `PruningBounds.lean`):
- Error = 0 at kept weights
- Error = |W_ij| at pruned weights
- Total error: `‖ΔW‖²_F = Σ(pruned weights)²`

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

PRUNED_OUTPUT = f'/content/models/{MODEL_CHOICE}-pruned-{PRUNING_SPARSITY*100:.0f}pct'

if PRUNING_SPARSITY > 0:
    print(f'✂️ Pruning with {PRUNING_SPARSITY*100:.0f}% sparsity...')

    tokenizer = AutoTokenizer.from_pretrained(QUANT_OUTPUT, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        QUANT_OUTPUT, torch_dtype=torch.float16,
        device_map='auto', trust_remote_code=True,
    )

    pruned_total = 0
    param_total = 0

    for name, param in model.named_parameters():
        if 'weight' in name and param.dim() == 2:
            with torch.no_grad():
                scores = param.abs()
                threshold = torch.quantile(
                    scores.flatten().float(), PRUNING_SPARSITY
                )
                mask = scores > threshold
                param.data *= mask.to(param.dtype)
                pruned_total += (~mask).sum().item()
                param_total += param.numel()

    actual_sparsity = pruned_total / max(param_total, 1)
    print(f'   Pruned {pruned_total:,} / {param_total:,} params')
    print(f'   Actual sparsity: {actual_sparsity*100:.1f}%')

    os.makedirs(PRUNED_OUTPUT, exist_ok=True)
    model.save_pretrained(PRUNED_OUTPUT)
    tokenizer.save_pretrained(PRUNED_OUTPUT)
    CURRENT_MODEL = PRUNED_OUTPUT
else:
    print('Pruning disabled.')
    CURRENT_MODEL = QUANT_OUTPUT

print(f'\n✅ Current model: {CURRENT_MODEL}')

## 5. Distill (Optional)

Distill from 7B teacher to 1.5B student.

**Theory** (from `DistillationTheory.lean`):
- Soft targets: `softTarget(z, T) = exp(z/T)`
- Higher T → softer distribution (`higher_temp_softer`)
- EML student: 4Ld params vs Ld² (`eml_student_compact`)

In [ ]:
if DISTILL_ENABLED:
    import torch.nn.functional as F
    from transformers import TrainingArguments, Trainer
    from datasets import load_dataset

    STUDENT_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
    DISTILL_OUTPUT = f'/content/models/{MODEL_CHOICE}-distilled'
    T = 4.0  # Temperature
    ALPHA = 0.5  # Distillation weight

    print(f'🎓 Distilling {MODEL_ID} → {STUDENT_ID}')
    print(f'   Temperature: {T}, Alpha: {ALPHA}')

    tokenizer = AutoTokenizer.from_pretrained(CURRENT_MODEL, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    teacher = AutoModelForCausalLM.from_pretrained(
        CURRENT_MODEL, torch_dtype=torch.float16,
        device_map='auto', trust_remote_code=True,
    )
    teacher.eval()

    student = AutoModelForCausalLM.from_pretrained(
        STUDENT_ID, torch_dtype=torch.float16,
        device_map='auto', trust_remote_code=True,
    )

    dataset = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')
    dataset = dataset.filter(lambda x: len(x['text']) > 50)
    dataset = dataset.select(range(min(len(dataset), 5000)))

    def tokenize(examples):
        return tokenizer(examples['text'], truncation=True, max_length=512, padding='max_length')

    dataset = dataset.map(tokenize, batched=True, remove_columns=['text'])
    dataset.set_format('torch')

    class DistillTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            out_s = model(**inputs)
            with torch.no_grad():
                out_t = teacher(**inputs)
            soft_s = F.log_softmax(out_s.logits / T, dim=-1)
            soft_t = F.softmax(out_t.logits / T, dim=-1)
            kd_loss = F.kl_div(soft_s, soft_t, reduction='batchmean') * (T * T)
            hard_loss = out_s.loss if out_s.loss is not None else 0
            loss = ALPHA * kd_loss + (1 - ALPHA) * hard_loss
            return (loss, out_s) if return_outputs else loss

    args = TrainingArguments(
        output_dir=DISTILL_OUTPUT, num_train_epochs=3,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        learning_rate=2e-5, fp16=True, save_strategy='epoch',
        logging_steps=50, report_to='none',
    )
    trainer = DistillTrainer(model=student, args=args, train_dataset=dataset)
    trainer.train()

    os.makedirs(DISTILL_OUTPUT, exist_ok=True)
    student.save_pretrained(DISTILL_OUTPUT)
    tokenizer.save_pretrained(DISTILL_OUTPUT)
    CURRENT_MODEL = DISTILL_OUTPUT
    print(f'\n✅ Distillation complete: {DISTILL_OUTPUT}')
else:
    print('Distillation disabled. Using compressed model directly.')

## 6. Benchmark & Telemetry

In [ ]:
import math
from datasets import load_dataset

print('📊 Running benchmarks...')
print('=' * 60)

# --- Perplexity ---
tokenizer = AutoTokenizer.from_pretrained(CURRENT_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    CURRENT_MODEL, torch_dtype=torch.float16,
    device_map='auto', trust_remote_code=True,
)
model.eval()

test_data = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
text = '\n\n'.join([t for t in test_data['text'] if len(t) > 0])
enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=2048)
input_ids = enc.input_ids.to(model.device)

with torch.no_grad():
    out = model(input_ids, labels=input_ids)
    loss = out.loss.item()
ppl = math.exp(loss)
print(f'Perplexity: {ppl:.2f} (loss={loss:.4f})')

# --- Inference Speed ---
prompt = 'Explain quantum computing in simple terms.'
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

# Warmup
with torch.no_grad():
    model.generate(**inputs, max_new_tokens=10)

torch.cuda.reset_peak_memory_stats()
start = time.time()
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=256, do_sample=False)
elapsed = time.time() - start

new_tokens = output.shape[1] - inputs['input_ids'].shape[1]
tps = new_tokens / elapsed
peak_vram = torch.cuda.max_memory_allocated() / (1024**3)

print(f'Tokens/sec: {tps:.1f}')
print(f'Time to generate {new_tokens} tokens: {elapsed:.2f}s')
print(f'Peak VRAM: {peak_vram:.2f} GB')
print(f'Model size: {dir_size_gb(CURRENT_MODEL):.2f} GB')

# --- Theoretical vs Measured ---
delta = 2.0 / (2 ** QUANT_BITS)
predicted_ppl_factor = math.exp(delta / 2)
print(f'\n📐 Compression Theory Check:')
print(f'   Predicted PPL factor: ≤ {predicted_ppl_factor:.4f}')
print(f'   Quant compression: {16/QUANT_BITS:.1f}×')
if PRUNING_SPARSITY > 0:
    print(f'   Pruning compression: {1/(1-PRUNING_SPARSITY):.1f}×')
    total_ratio = (16/QUANT_BITS) * (1/(1-PRUNING_SPARSITY))
else:
    total_ratio = 16/QUANT_BITS
print(f'   Total theoretical compression: {total_ratio:.1f}×')

# --- Telemetry ---
telemetry = {
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'model': MODEL_CHOICE,
    'quant': f'{QUANT_METHOD}-{QUANT_BITS}bit',
    'pruning_sparsity': PRUNING_SPARSITY,
    'distilled': DISTILL_ENABLED,
    'perplexity': round(ppl, 2),
    'tokens_per_second': round(tps, 1),
    'peak_vram_gb': round(peak_vram, 2),
    'model_size_gb': round(dir_size_gb(CURRENT_MODEL), 2),
    'compression_ratio': round(total_ratio, 2),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
}

# Save telemetry
os.makedirs('/content/telemetry', exist_ok=True)
with open('/content/telemetry/benchmark.jsonl', 'a') as f:
    f.write(json.dumps(telemetry) + '\n')

# Also save to Google Drive
gdrive_telem = '/content/drive/MyDrive/qwen_cache/telemetry.jsonl'
os.makedirs(os.path.dirname(gdrive_telem), exist_ok=True)
with open(gdrive_telem, 'a') as f:
    f.write(json.dumps(telemetry) + '\n')

print(f'\n📡 Telemetry saved.')
print(json.dumps(telemetry, indent=2))

## 7. Interactive Inference

Run the optimized model interactively.

In [ ]:
def chat(prompt, max_tokens=256):
    """Generate a response from the optimized model."""
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)

    start = time.time()
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=max_tokens,
            do_sample=True, temperature=0.7, top_p=0.9,
        )
    elapsed = time.time() - start

    response = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    new_tokens = output.shape[1] - inputs['input_ids'].shape[1]
    print(f'[{new_tokens} tokens in {elapsed:.1f}s = {new_tokens/elapsed:.1f} tok/s]')
    return response

# Try it!
response = chat('What is the most beautiful equation in mathematics and why?')
print(response)

## 8. Save to Google Drive

Checkpoint the optimized model for future sessions.

In [ ]:
SAVE_PATH = f'/content/drive/MyDrive/qwen_cache/{MODEL_CHOICE}-optimized'
print(f'💾 Saving optimized model to Google Drive...')
os.makedirs(SAVE_PATH, exist_ok=True)
shutil.copytree(CURRENT_MODEL, SAVE_PATH, dirs_exist_ok=True)
print(f'✅ Saved to {SAVE_PATH} ({dir_size_gb(SAVE_PATH):.2f} GB)')